In [ ]:
# import libraries
# Import libraries. You may or may not use all of these.
!pip install -q git+https://github.com/tensorflow/docs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
  # %tensorflow_version only exists in Colab.
  %tensorflow_version 2.x
except Exception:
  pass
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

!pip install tensorflow-datasets
import tensorflow_datasets as tfds

print(tf.__version__)

# New Section

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

In [ ]:
# Load the data
train_df = pd.read_csv(train_file_path, sep='\t', header=None, names=['label', 'message'])
test_df = pd.read_csv(test_file_path, sep='\t', header=None, names=['label', 'message'])
print(train_df.head)
print(test_df.head)

In [ ]:
train_labels = train_df['label'].apply(lambda x: 1 if x == 'spam' else 0).values
test_labels = test_df['label'].apply(lambda x: 1 if x == 'spam' else 0).values
print(train_labels)
print(test_labels)

In [ ]:
train_messages = train_df['message'].values
test_messages = test_df['message'].values
print(train_messages)
print(test_messages)

In [ ]:
# Text vectorization layer
VOCAB_SIZE = 1000
MAX_LEN = 100

encoder = layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_sequence_length=MAX_LEN
)
encoder.adapt(train_messages)
print(encoder)
vocab = encoder.get_vocabulary()
print(len(vocab))      # should be <= 1000
print(vocab[:20])


In [ ]:
sample = encoder(train_messages[:10])

print(np.isnan(sample.numpy()).any())
print(sample.numpy().dtype)

In [ ]:

# Build the model
vocab_size = len(encoder.get_vocabulary())
model = keras.Sequential([
   #  keras.Input(shape=(1,), dtype=tf.string),
    encoder,
    layers.Embedding(input_dim=vocab_size+1, output_dim=16,mask_zero=False),
    layers.GlobalAveragePooling1D(),
    layers.Dense(24, activation='relu'),
    #layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)
# Build the model by running one forward pass, then show summary
model(train_messages[:1])

model.summary()

sample = encoder(train_messages[:5])
print(sample.numpy())
print(sample.numpy().min(), sample.numpy().max())

In [ ]:
# Check for NaN before any training
test_out = model(train_messages[:5])
print(test_out.numpy())
print(np.isnan(test_out.numpy()).any())

In [ ]:
sample = encoder(train_messages[:10])

print(np.isnan(sample.numpy()).any())
print(sample.numpy().dtype)

In [ ]:
print(train_df['message'].isna().sum())
print((train_df['message'].str.strip() == '').sum())
print(train_labels.dtype, test_labels.dtype)
print(np.isnan(train_labels).sum(), np.isnan(test_labels).sum())


In [ ]:
# Train the model
history = model.fit(
    train_messages,
    train_labels,
    epochs=5,
    batch_size=32,
    validation_data=(test_messages, test_labels),
    verbose=2
)
print(history.history)

In [ ]:
print(tf.__version__)
print(train_labels[:10])
print(train_messages[:5])

In [ ]:
print(train_messages.dtype)
print(set(type(x) for x in train_messages[:20]))

In [ ]:
# Check encoder output
sample = encoder(train_messages[:5])
print(sample.numpy())
print(sample.numpy().min(), sample.numpy().max())



In [ ]:
# Check model output before training
test_out = model(train_messages[:5])
print(test_out.numpy())
#print(np.isnan(test_out.numpy()).any())

In [ ]:
print(encoder.vocabulary_size())
print(VOCAB_SIZE)

sample = encoder(train_messages[:5])
print(sample.numpy())
print(sample.numpy().min(), sample.numpy().max())

In [ ]:
# Plot training history
import tensorflow_docs.plots

plotter = tensorflow_docs.plots.HistoryPlotter(metric='accuracy')
plotter.plot({'Model': history})
plt.show()

In [ ]:
def predict_message(pred_text):
  # Convert the input text to a TensorFlow Tensor
  input_tensor = tf.constant([pred_text])
  print(input_tensor)
  pred = model.predict(input_tensor, verbose=1)
  print(pred)
  prob = float(pred[0][0])
  label = 'spam' if prob >= 0.05 else 'ham'
  return [prob, label]

pred_text = "how are you doing today?"

prediction = predict_message(pred_text)
print(prediction)

In [ ]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    #print(prediction)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()
